# GENESIS Phase 0: Train 50M HLRT Model

End-to-end training of the 41M parameter HLRT model on Colab Pro.

**What this does:**
1. Mounts Google Drive (checkpoints survive disconnects)
2. Clones the repo and installs dependencies
3. Streams training data from HuggingFace (no big download)
4. Trains the 3-tier HLRT model with full logging
5. Saves checkpoints + best model to Drive

**Requirements:** Colab Pro with T4/A100/L4 GPU (~4-6GB VRAM needed)

**Time estimates:**
- Data prep: ~30-60 min (streaming 1B tokens)
- Training 10k steps: ~3 hrs (A100) / ~10 hrs (T4)
- Chinchilla-optimal (3800 steps): ~1 hr (A100) / ~4 hrs (T4)

## 1. GPU Check + Google Drive Mount

In [ ]:
import torch

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    if vram_gb >= 30:
        print("A100 detected — training will be fast (~3 hrs)")
    elif vram_gb >= 20:
        print("L4 detected — training will take ~5-8 hrs")
    else:
        print("T4 detected — training will take ~10-15 hrs")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU")
    print("Training on CPU will be extremely slow.")

In [ ]:
# Mount Google Drive so checkpoints survive disconnects
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory on Drive
import os
CHECKPOINT_DIR = '/content/drive/MyDrive/genesis_checkpoints/phase0'
DATA_DIR = '/content/data/packed_phase0'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

## 2. Install Dependencies

In [ ]:
# Clone the repo
!git clone https://github.com/brendenk6/LLM-Research-.git /content/genesis 2>/dev/null || echo "Repo already cloned"
%cd /content/genesis
!git pull origin main

# Install the project + data dependencies
!pip install -e ".[dev]" -q
!pip install datasets tiktoken pyyaml -q

print("\nDependencies installed!")

## 3. Prepare Training Data

This streams from HuggingFace's FineWeb-Edu dataset. No full download needed.

**Choose your token budget:**
- `100M` tokens: Quick smoke test (~5 min prep, ~1 hr train)
- `500M` tokens: Decent training run (~20 min prep, ~5 hrs train on T4)
- `1.1B` tokens: Chinchilla-optimal (~45 min prep, ~10 hrs train on T4)

You can always add more data later and resume training.

In [ ]:
#@title Data Preparation Settings
NUM_TOKENS = 500000000  #@param [100000000, 500000000, 1100000000] {type:"raw"}
# 100M = smoke test, 500M = good first run, 1.1B = Chinchilla-optimal

import time

# Check if data already exists (from a previous run)
meta_path = os.path.join(DATA_DIR, 'meta.json')
if os.path.exists(meta_path):
    import json
    with open(meta_path) as f:
        meta = json.load(f)
    existing_tokens = meta['total_tokens']
    print(f"Found existing data: {existing_tokens/1e6:.0f}M tokens")
    if existing_tokens >= NUM_TOKENS * 0.9:  # close enough
        print("Skipping data prep — already have enough data!")
        SKIP_PREP = True
    else:
        print(f"Need more data ({NUM_TOKENS/1e6:.0f}M requested). Re-running prep...")
        SKIP_PREP = False
else:
    print(f"No existing data found. Will prepare {NUM_TOKENS/1e6:.0f}M tokens.")
    SKIP_PREP = False

In [ ]:
if not SKIP_PREP:
    start = time.time()
    !python -m genesis.training.prepare_data \
        --dataset HuggingFaceFW/fineweb-edu \
        --subset sample-10BT \
        --out-dir {DATA_DIR} \
        --seq-len 2048 \
        --num-tokens {NUM_TOKENS} \
        --val-ratio 0.01 \
        --vocab-size 32000
    elapsed = time.time() - start
    print(f"\nData prep took {elapsed/60:.1f} minutes")
else:
    print("Using existing data.")

# Verify
import json
with open(os.path.join(DATA_DIR, 'meta.json')) as f:
    meta = json.load(f)
print(f"\nDataset ready:")
print(f"  Train blocks: {meta['train_blocks']:,}")
print(f"  Val blocks:   {meta['val_blocks']:,}")
print(f"  Total tokens: {meta['total_tokens']/1e6:.0f}M")
print(f"  Block size:   {meta['block_size']}")

## 4. Build Model + Verify

In [ ]:
import yaml
from genesis.training.train_phase0 import build_model
from genesis.model.tier_composer import count_parameters

# Load config
with open('genesis/training/configs/phase0_50m.yaml') as f:
    cfg = yaml.safe_load(f)

# Build model
model = build_model(cfg)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Parameter breakdown
params = count_parameters(model)
print(f"\nModel parameter breakdown:")
for k, v in params.items():
    print(f"  {k:12s}: {v/1e6:.2f}M")

# Quick forward pass test
model.eval()
test_ids = torch.randint(0, cfg['model']['vocab_size'], (2, 64), device=device)
with torch.no_grad():
    out = model(test_ids)
print(f"\nForward pass OK: logits shape = {out['logits'].shape}")

# VRAM usage
if torch.cuda.is_available():
    mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"VRAM after model load: {mem:.2f} GB")

## 5. Train!

This runs the full training loop. Key things to watch:
- **Loss** should drop steadily for the first ~2000 steps
- **Val loss** tracks training loss (no overfitting at this scale)
- **Gate scores** show tier routing is working
- Checkpoints save every 500 steps + best model auto-saved

If Colab disconnects, just re-run from Cell 1 — it will detect existing data and resume from the latest checkpoint.

In [ ]:
#@title Training Settings
MAX_STEPS = 10000  #@param {type:"integer"}
RESUME_FROM = ""  #@param {type:"string"}
# Leave RESUME_FROM empty for fresh start.
# To resume: set to path like '/content/drive/MyDrive/genesis_checkpoints/phase0/checkpoint_step2000.pt'

# Auto-detect latest checkpoint for resume
if not RESUME_FROM:
    import glob
    existing_ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'checkpoint_step*.pt')))
    if existing_ckpts:
        RESUME_FROM = existing_ckpts[-1]
        print(f"Found existing checkpoint, will resume from: {RESUME_FROM}")
    else:
        print("Starting fresh training run.")

# Override max_steps in config
cfg['training']['max_steps'] = MAX_STEPS

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

# Override paths for Colab
cfg['data']['packed_dir'] = DATA_DIR

from genesis.training.train_phase0 import train

train(
    cfg,
    resume_path=RESUME_FROM if RESUME_FROM else None,
    checkpoint_dir=CHECKPOINT_DIR,
)

## 6. Check Results

In [ ]:
import glob

# List saved checkpoints
ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, '*.pt')))
print(f"Saved checkpoints ({len(ckpts)} total):")
for ckpt in ckpts:
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f"  {os.path.basename(ckpt):40s} ({size_mb:.1f} MB)")

# Load best model and check it works
best_path = os.path.join(CHECKPOINT_DIR, 'best.pt')
if os.path.exists(best_path):
    from genesis.model.hlrt import HLRT
    best_ckpt = torch.load(best_path, map_location=device, weights_only=False)
    best_model = build_model(cfg).to(device)
    best_model.load_state_dict(best_ckpt['model_state_dict'])
    best_model.eval()

    with torch.no_grad():
        out = best_model(test_ids, return_tier_activations=True)
        print(f"\nBest model (step {best_ckpt['step']}):")
        print(f"  Logits shape: {out['logits'].shape}")
        if 'gate1_scores' in out.get('tier_activations', {}):
            g1 = out['tier_activations']['gate1_scores']
            print(f"  Gate 1 mean score: {g1.mean().item():.4f}")
            print(f"  Tier 2 escalation rate: {(g1 > 0.3).float().mean().item():.1%}")
    print(f"\nBest model loaded successfully from Drive!")
    print(f"Path: {best_path}")
else:
    print("No best.pt found — training may not have run eval yet.")

## 7. Save Tiers for LEGO Composition (Optional)

Save individual tiers so you can later compose them with tiers from larger models.

In [ ]:
from genesis.model.tier_composer import save_tier

# Use the best model if available, otherwise final
save_model = best_model if 'best_model' in dir() else model

tier_dir = os.path.join(CHECKPOINT_DIR, 'tiers')
os.makedirs(tier_dir, exist_ok=True)

for tier_id in [1, 2, 3]:
    save_tier(save_model, tier_id, os.path.join(tier_dir, f'tier{tier_id}_50m.pt'))

print(f"\nTier checkpoints saved to: {tier_dir}")
print("You can compose these with tiers from larger models using TierComposer.")
print("\nExample:")
print("  from genesis.model.tier_composer import TierComposer")
print("  composer = TierComposer(target_config)")
print(f"  composer.load_tier(1, '{tier_dir}/tier1_50m.pt')")
print("  composer.load_tier(2, 'path/to/100m/tier2.pt')")
print("  model = composer.build(freeze_tiers=[1])")

## What's Next?

After Phase 0 training completes:

1. **Check the loss curve** — if it dropped significantly and plateaued, the architecture works
2. **Check gate behavior** — Tier 2 should activate on ~30-40% of chunks, Tier 3 on ~5-10%
3. **Scale up** — Train a 100M model, then LEGO the best tiers together:
   ```python
   composer = TierComposer(config_100m)
   composer.load_tier(1, 'tiers/tier1_50m.pt')   # reuse from 50M
   composer.load_tier(2, 'tiers/tier2_100m.pt')   # from 100M
   model = composer.build(freeze_tiers=[1])        # only train the new parts
   ```
4. **Phase 1** — Move to the full 1B config on Lambda when ready